# Etapa 2 — Modelagem e Avaliação
## Telco Customer Churn — IBM dataset

Continuação da Etapa 1 (`eda_customer_churn.ipynb`). Aqui comparamos três famílias de modelo — Regressão Logística (baseline), Random Forest (ensemble de árvores) e MLPClassifier (rede neural) — usando validação cruzada, e escolhemos um modelo campeão.


- Reaproveitamos exatamente a mesma limpeza/seleção de features da Etapa 1, para que a comparação entre modelos seja justa e os números da Regressão Logística aqui batam com o baseline já registrado.
- `churn_score` e `churn_reason` continuam de fora do conjunto de features: `churn_score` é a saída de outro modelo preditivo (vazamento de dados) e `churn_reason` só existe para quem já deu churn.
- Diferença metodológica em relação à Etapa 1: o `StandardScaler` agora vive dentro de um `Pipeline`/`ColumnTransformer`, então cada fold da validação cruzada ajusta sua própria escala — evita um vazamento sutil (fit do scaler no treino inteiro antes do CV).


# Setup

In [ ]:
import numpy as np
import pandas as pd

import mlflow

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

import joblib


In [ ]:
mlflow.set_tracking_uri('http://localhost:5000')
mlflow.set_experiment('telco_customer_churn')

pd.set_option('display.max_columns', None)

RANDOM_STATE = 42


# Carregamento e preparação dos dados

Mesmos passos da Etapa 1: leitura do Excel, renomeação para snake_case, correção de `total_charges` e seleção do conjunto de features do baseline (sem colunas geográficas, `churn_score` ou `churn_reason`).

In [ ]:
df = pd.read_excel('../data/Telco_customer_churn.xlsx')
df.rename(columns=lambda x: x.replace(' ', '_').lower(), inplace=True)

if df['total_charges'].dtype.name == 'object':
    total_charges_numeric = pd.to_numeric(df['total_charges'], errors='coerce')
    mediana = total_charges_numeric.median()
    mask = df['total_charges'].str.contains(' ', na=False)
    df.loc[mask, 'total_charges'] = mediana
    df['total_charges'] = df['total_charges'].astype(float)

df.shape


In [ ]:
feature_cols = [
    'gender', 'senior_citizen', 'partner', 'dependents',
    'tenure_months', 'phone_service', 'multiple_lines', 'internet_service',
    'online_security', 'online_backup', 'device_protection', 'tech_support',
    'streaming_tv', 'streaming_movies', 'contract', 'paperless_billing',
    'payment_method', 'monthly_charges', 'total_charges', 'cltv', 'churn_value',
]

df = df[feature_cols].copy()

df['multiple_lines'] = df['multiple_lines'].replace('No phone service', 'No')

dependentes = [
    'online_security', 'online_backup', 'device_protection', 'tech_support',
    'streaming_tv', 'streaming_movies',
]
df[dependentes] = df[dependentes].replace('No internet service', 'No')

df.head()


In [ ]:
cat_cols = [
    'gender', 'senior_citizen', 'partner', 'dependents',
    'phone_service', 'multiple_lines', 'internet_service', 'online_security',
    'online_backup', 'device_protection', 'tech_support', 'streaming_tv',
    'streaming_movies', 'contract', 'paperless_billing', 'payment_method',
]

df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True, dtype='int')

X = df_encoded.drop(columns='churn_value')
y = df_encoded['churn_value']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

X_train.shape, X_test.shape


# Pipeline de pré-processamento

Apenas as 4 colunas numéricas originais precisam de escala; as demais já são dummies 0/1. O `ColumnTransformer` aplica o `StandardScaler` só nelas e passa o resto direto (`remainder='passthrough'`). Esse pré-processador entra dentro do `Pipeline` de cada modelo, então é ajustado de novo em cada fold do CV e, no fim, no treino completo.

In [ ]:
scale_cols = ['tenure_months', 'monthly_charges', 'total_charges', 'cltv']

preprocessor = ColumnTransformer(
    transformers=[('scale', StandardScaler(), scale_cols)],
    remainder='passthrough',
)


# Modelos candidatos

- **Regressão Logística**: mesma configuração da Etapa 1 (parâmetros padrão), usada aqui como referência de comparação.
- **Random Forest**: `class_weight='balanced'` para compensar o desbalanceamento (~27% de churn) — sem isso o modelo tende a favorecer a classe majoritária e perder recall. `max_depth=12` limita o tamanho das árvores (sem isso o artefato salvo passava de 50MB) e ajuda a controlar overfitting.
- **MLPClassifier**: `MLPClassifier` não aceita `class_weight`; se o recall ficar baixo, a saída seria calibrar o threshold de decisão a partir de `predict_proba` (não feito aqui, fica como próximo passo).

In [ ]:
models = {
    'logistic_regression': LogisticRegression(random_state=RANDOM_STATE, max_iter=1000),
    'random_forest': RandomForestClassifier(
        n_estimators=300, max_depth=12, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1
    ),
    'mlp': MLPClassifier(
        hidden_layer_sizes=(64, 32), max_iter=500, early_stopping=True, random_state=RANDOM_STATE
    ),
}

pipelines = {
    name: Pipeline([('preprocess', preprocessor), ('clf', model)])
    for name, model in models.items()
}


# Validação cruzada

`StratifiedKFold` com 5 folds preserva a proporção de churn em cada fold. Reportamos média ± desvio padrão de F1 e ROC-AUC para checar se o resultado é estável, não sorte de um único split.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

cv_results = {}

for name, pipeline in pipelines.items():
    scores = cross_validate(pipeline, X_train, y_train, cv=cv, scoring=scoring)
    cv_results[name] = {
        f'cv_{metric}_mean': scores[f'test_{metric}'].mean() for metric in scoring
    }
    cv_results[name].update({
        f'cv_{metric}_std': scores[f'test_{metric}'].std() for metric in scoring
    })
    print(f"{name}: F1 = {cv_results[name]['cv_f1_mean']:.4f} +/- {cv_results[name]['cv_f1_std']:.4f} | "
          f"ROC-AUC = {cv_results[name]['cv_roc_auc_mean']:.4f} +/- {cv_results[name]['cv_roc_auc_std']:.4f}")


# Avaliação no conjunto de teste e registro no MLflow

In [ ]:
test_results = {}

for name, pipeline in pipelines.items():
    with mlflow.start_run(run_name=f'{name}_etapa2'):
        pipeline.fit(X_train, y_train)

        mlflow.log_params({f'clf__{k}': v for k, v in models[name].get_params().items()})
        mlflow.log_metrics(cv_results[name])

        y_pred = pipeline.predict(X_test)
        y_prob = pipeline.predict_proba(X_test)[:, 1]

        metrics = {
            'test_accuracy': accuracy_score(y_test, y_pred),
            'test_precision': precision_score(y_test, y_pred),
            'test_recall': recall_score(y_test, y_pred),
            'test_f1': f1_score(y_test, y_pred),
            'test_roc_auc': roc_auc_score(y_test, y_prob),
        }
        mlflow.log_metrics(metrics)
        mlflow.sklearn.log_model(pipeline, name=name, serialization_format='cloudpickle')

        test_results[name] = metrics

test_results


# Tabela comparativa

In [ ]:
comparison = pd.DataFrame({
    name: {**cv_results[name], **test_results[name]} for name in models
}).T

comparison = comparison[[
    'test_accuracy', 'test_precision', 'test_recall', 'test_f1', 'test_roc_auc',
    'cv_f1_mean', 'cv_f1_std', 'cv_roc_auc_mean', 'cv_roc_auc_std',
]].round(4)

comparison.to_csv('../models/model_comparison.csv')
comparison


**Resultado observado** (com `max_depth=12` no Random Forest):

| Modelo | Test Acc | Test Prec | Test Recall | Test F1 | Test ROC-AUC | CV F1 (média ± dp) | CV ROC-AUC (média ± dp) |
|---|---|---|---|---|---|---|---|
| Logistic Regression | 0.8034 | 0.6456 | 0.5749 | 0.6082 | 0.8490 | 0.6196 ± 0.0322 | 0.8588 ± 0.0125 |
| Random Forest | 0.7722 | 0.5521 | 0.7513 | **0.6365** | 0.8499 | **0.6445 ± 0.0218** | 0.8578 ± 0.0106 |
| MLP | 0.8013 | 0.6320 | 0.6016 | 0.6164 | 0.8505 | 0.5934 ± 0.0339 | 0.8581 ± 0.0128 |

- **Random Forest** vence em F1 (0.6365 no teste, 0.6445 ± 0.0218 no CV) — o efeito do `class_weight='balanced'` aparece no recall bem mais alto (0.7513 contra 0.5749 da Regressão Logística e 0.6016 do MLP), ao custo de precisão mais baixa (0.5521). Limitar `max_depth=12` também baixou o desvio padrão do CV (0.0218, o menor dos três) e reduziu o artefato salvo de 53MB para 27MB.
- No ROC-AUC os três modelos praticamente empatam (0.8490–0.8505) — ou seja, a capacidade de *rankear* clientes por risco é parecida; a diferença de F1 vem de onde cada modelo posiciona o threshold de decisão, não de discriminação bruta.
- CV e teste concordam: Random Forest é o melhor em F1 nos dois. Regressão Logística e MLP trocam de posição entre CV (MLP pior) e teste (Regressão Logística pior), mas ficam sempre muito próximos um do outro — não é uma diferença que eu chamaria de significativa.
- Sem sinais de overfitting: os três têm desvio padrão de CV baixo (≤ 0.034) e métricas de teste alinhadas com a média do CV.
- **Conclusão**: dado que escolhemos F1 como critério de campeão (equilíbrio entre não perder churners e não gastar retenção à toa), o **Random Forest é o modelo campeão**. Se a prioridade de negócio fosse maximizar recall puro (não perder nenhum churner, mesmo com mais falsos positivos), a escolha seria a mesma; se fosse ROC-AUC para uma lista de risco ranqueada, os três modelos seriam praticamente equivalentes.


# Modelo campeão

In [ ]:
champion_name = max(test_results, key=lambda name: test_results[name]['test_f1'])
champion_pipeline = pipelines[champion_name]

print(f'Modelo campeão: {champion_name}')
print(test_results[champion_name])

joblib.dump(champion_pipeline, '../models/champion_model.joblib')


O `champion_model.joblib` salvo é o `Pipeline` completo (pré-processamento + modelo), então recebe as mesmas colunas dummy-encoded de `X` diretamente — não precisa reaplicar o `StandardScaler` manualmente como no `baseline_model.joblib` da Etapa 1.